# 11.1 維度相同就能理解圖片嗎？


把圖片接到文字模型後，紅圖與藍圖的回答分數不同，這看起來像模型在看圖。但是兩次都答「綠」也可能有不同分數；差異只表示訊號有傳過來，沒有保證內容正確。本節把接頭能傳遞差異與模型理解圖片分開檢查。

前置是[10.6的projector接頭](https://birdhackor.github.io/tiny-perceptron-vlm/10.6.html)與[10.5的視覺特徵訓練](https://birdhackor.github.io/tiny-perceptron-vlm/10.5.html)。維度相容指每位置數字數量符合下一層要求；語義對齊則指視覺內容經接頭後，能被語言模型用來作出正確回答。同樣8個數字，並不表示兩個系統對它們有同一種解讀。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
projector = nn.Linear(4, 8)
a, b = torch.zeros(1, 4), torch.ones(1, 4)
difference = (projector(a) - projector(b)).norm()
print("輸出形狀", tuple(projector(a).shape))
print("對輸入差異敏感", difference.item() > 0)

輸入是兩個人工特徵：全0與全1，都寬4。隨機接頭把它們轉成8維，`norm()`計算兩個輸出差向量的長度。輸出形狀`(1,8)`，第二行True。這個結果沒有標準答案，也沒有語言模型，所以只能說隨機變換通常會對輸入有反應。它甚至還不知道a代表紅還是方塊。

實際對齊至少需要三種對照。先用文字告訴語言模型「這是紅色圓形」，問顏色，確認它原本會答紅；再送相同屬性的圖，問同一問題，確認能答紅；最後換藍色同形狀圖，確認答案按規則改為藍。第一個檢查語言基礎，第二個檢查跨介面使用，第三個檢查圖片因果作用。只看兩組logits不同，會漏掉是否答對。

若編碼器本來無法區分紅藍，接頭沒有足夠資訊；若語言模型連文字描述都不會用，單獨訓練接頭也未必夠。對齊要把已有相關能力接起來，不能預設一個小接頭必定補齊兩端所有知識。後面會分別列出可訓練範圍與訓練目標。

可以把三種對照都問同一句「什麼顏色」：文字紅圓→紅，紅圓圖→紅，藍圓圖→藍。第一步若失敗，圖片接頭無法獨自證明後續錯誤來自哪裡；第二步失敗而第一步成功，就更值得查視覺特徵與接頭；第三步才檢查它沒有對每張圖都回紅。這是一個逐段定位方法，不是在測試前假設各組件都有效。隨機接頭對0與1的輸出差通常非零，就像插座有電卻不表示收音機調到正確頻道，必須再核對收到的內容。

練習在`difference = ...`那一行之前加上`with torch.no_grad(): projector.weight.zero_()`，保持偏置不變，然後從頭重跑。先預測兩個輸出都等於同一偏置，重新計算的差長度為0、第二行False，再執行核對。不能等差值已算好才清零權重，因為既有的`difference`不會自動重算。這個失敗能找出接頭沒有傳差異，卻仍不能獨立診斷語言理解是否正確；不同檢查負責不同問題。
